# Train CS224N GPT-2 SST With New Classifier Code

Run this notebook on Kaggle with GPU enabled. It pulls the latest project code, trains SST using the improved classifier, writes predictions, and builds the submission zip.

## 1. Install dependencies

In [ ]:
%pip uninstall -y -q torch torchvision torchaudio
%pip install -q --no-cache-dir torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121
%pip install -q transformers einops scikit-learn tqdm

## 2. Pull the latest GitHub code

In [ ]:
from pathlib import Path
import shutil
import subprocess

repo_url = "https://github.com/Mahmoudmoh-cse/CSE224-GPT2.git"
repo_dir = Path("/kaggle/working/cs224n_gpt")

if repo_dir.exists() and (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"], check=True)
elif repo_dir.exists():
    shutil.rmtree(repo_dir)
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

%cd /kaggle/working/cs224n_gpt
!git log --oneline -1

## 3. Verify the new classifier options

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

!python -m py_compile classifier.py
!python classifier.py --help | sed -n '1,80p'

## 4. Train SST

This is the main improved command. If Kaggle runs out of memory, change `--batch_size 8` to `--batch_size 4`.

In [ ]:
!python classifier.py --task sst --encoder_backend hf --epochs 5 --lr 1e-5 --head_lr 5e-4 --batch_size 8 --use_gpu

## 5. Check outputs

In [ ]:
!ls -lah checkpoints
!ls -lah predictions

## 6. Build submission zip

In [ ]:
!python prepare_submit.py
!ls -lah /kaggle/working/*.zip